In [ ]:
import os
import sys
import matplotlib.pyplot as plt
import torch
from sklearn.metrics import confusion_matrix, f1_score, classification_report
import numpy as np
PROJECT_ROOT = os.path.abspath("..") 
sys.path.append(PROJECT_ROOT)

In [ ]:

from dataset.otto_final import TraceOttoDataset
from model.trace import TRACE
from utils.SplitData import split_data_Train_Val_Test
from utils.feature_engineering import get_between_features, get_elapsed_feature
from utils.plot_confussion_matrix import plot_confusion_matrix
from utils.training_utils import update_binary_metrics, initialize_TRACE_model
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")



In [ ]:
dataset_processed  = TraceOttoDataset(
    file_name='../train.jsonl',
    input_seq_len=64,
    min_timestamps_per_sample=32,
)

In [ ]:
trace_model = initialize_TRACE_model(dataset_processed=dataset_processed, num_classes=1, device=device)

checkpoint = torch.load("Model_TRACE_ATC_FinalVersion_SingleTask.pt", weights_only=False, map_location=device)
trace_model.load_state_dict(checkpoint["model_state_dict"])
best_thr = checkpoint["best_global_threshold"]

print(f"Loaded model Optimum threshold: {best_thr:.4f}")

In [ ]:
train_loader, validation_loader, test_loader = split_data_Train_Val_Test(dataset_processed, batch_size=128)


print(len(train_loader.dataset))
print(len(validation_loader.dataset))
print(len(test_loader.dataset))


In [ ]:
trace_model.eval()
criterion = torch.nn.BCEWithLogitsLoss()
test_loss = 0.0
test_correct = 0
test_total = 0

all_y_true = []
all_y_pred = []
with torch.no_grad():
    for inputs_test, targets_test in test_loader:
        label_test_task = targets_test["ATC"].unsqueeze(1).to(device)

        inputs_test = {k: v.to(device) for k, v in inputs_test.items()}

        delta_elapsed = get_elapsed_feature(inputs_test["timestamps"]).to(device)
        delta_between = get_between_features(inputs_test["timestamps"]).to(device)

        logits = trace_model(
            inputs_test["aid"],
            inputs_test["type"],
            delta_elapsed,
            delta_between
        )

        loss = criterion(logits, label_test_task.float())
        test_loss += loss.item()

        probs = torch.sigmoid(logits)
        preds = (probs >= best_thr).float()

        test_correct += (preds == label_test_task).sum().item()
        test_total += label_test_task.numel()

        all_y_true.append(label_test_task.cpu())
        all_y_pred.append(preds.cpu())


test_loss /= len(test_loader)
test_accuracy = test_correct / max(test_total, 1)


y_true = torch.cat(all_y_true).numpy().astype(int).ravel()
y_pred = torch.cat(all_y_pred).numpy().astype(int).ravel()

f1 = f1_score(y_true, y_pred)

cm = confusion_matrix(y_true, y_pred)

print(f"Test loss: {test_loss:.4f}")
print(f"Test accuracy: {test_accuracy:.4f}")
print(f"F1 Score: {f1}")
print(f"Optimum Threshold {best_thr:.4f}")
print("Confusion Matrix:\n", cm)
print("\nClassification Report:")
print(classification_report(y_true, y_pred, target_names=["0", "1"]))


fig = plot_confusion_matrix(
    cm,
    name_task="ATC",
    name_classes=["class 0", "class 1"]
)
plt.show()
plt.close(fig)
